#Setup Cells

In [ ]:
# Daily setup cell

import os
from google.colab import drive
from google.colab import userdata

# Mount Google Drive
drive.mount('/content/drive/')

# --- CONFIGURATION ---
project_folder = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/"
repo_owner = "LinoyHarush"
repo_name = "MedSAM_enh"
# ---------------------

# Securely fetch the user's personal PAT
try:
    pat = userdata.get('GITHUB_PAT')
except userdata.SecretNotFoundError:
    print("Error: 'GITHUB_PAT' not found in Colab Secrets.")
    print("Please add it via the key icon on the left sidebar before running.")
    raise

repo_url = f"https://{pat}@github.com/{repo_owner}/{repo_name}.git"
repo_path = os.path.join(project_folder, repo_name)

# Navigate to the working folder
os.makedirs(project_folder, exist_ok=True)
%cd {project_folder}

# Clone or Pull
if not os.path.isdir(repo_name):
    print(f"Cloning {repo_name} into Drive for the first time...")
    !git clone {repo_url}
else:
    print(f"Directory {repo_name} exists. Updating repository...")
    %cd {repo_name}
    # Update the origin URL so Git uses the PAT of whoever is running this cell right now
    !git remote set-url origin {repo_url}
    !git pull
    %cd ..

# Enter the repository for the remainder of the session
%cd {repo_name}

In [ ]:
# Install the packages and their dependencies
%cd /content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh/MedSAM/
!pip install -e .

In [ ]:
# Local git identity for this session so commits have Shelly's name
!git config --global user.name "<your-name>"
!git config --global user.email "<your-email>"

In [ ]:
# Local git identity for this session so commits have Linoy's name
!git config --global user.name "<your-name>"
!git config --global user.email "<your-email>"

In [ ]:
# copy the code and the model to the runtime's disk, check the GPU

%cd /content
REPO_D = "/content/drive/MyDrive/Deep_Learning_Course/MedSAM/MedSAM_enh"   # the only line to edit if the shared folder sits elsewhere in your Drive
!mkdir -p /content/code/MedSAM /content/work/data /content/work/experiments /content/train_ready_npz /content/runs
!rsync -a --exclude='work_dir' --exclude='__pycache__' "{REPO_D}/MedSAM/" /content/code/MedSAM/
!grep -c "def board" /content/code/MedSAM/colab_runs.py
!cp "{REPO_D}/MedSAM/work_dir/LiteMedSAM/lite_medsam.pth" /content/lite_medsam.pth
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")

In [ ]:
# load the helpers and copy the data to the runtime's disk

import sys; sys.path.insert(0, "/content/code/MedSAM")
import colab_runs as cr          # MedSAM/colab_runs.py: train, score, save, compare, board, ...
cr.init(REPO_D)
cr.copy_data()                   # retries and reconnects Drive by itself if the mount drops

In [ ]:
# R3 train (LoRA decoder q,v, r=8)
run = cr.train("R3", 0)

In [ ]:
# R3 · Score, save, compare with Z

run = "R3_s0"
cr.score(run, cr.infer_ckpt(run))
cr.save(run)
cr.compare(run)
cr.board()          # shows Shelly's runs too, once she has saved them

In [ ]:
# R1 · Train (Full fine-tuning)

run = cr.train("R1", 0)

In [ ]:
# R1 · Score, save, compare with Z

run = "R1_s0"
cr.score(run, cr.infer_ckpt(run))
cr.save(run)
cr.compare(run)
cr.board()

In [ ]:
#LoRA encoder qkv only, only if R6_s0 > R3_s0

run = cr.train("R7", 0)

In [ ]:
# R7 · Score, save, compare with Z

run = "R7_s0"
cr.score(run, cr.infer_ckpt(run))
cr.save(run)
cr.compare(run)
cr.board()

Train Seed 2 for R1 and R7

In [ ]:
# train R7 seed 2 (LoRA on the encoder only)
run = cr.train("R7", 2)

In [ ]:
run = "R7_s2"
cr.score(run, cr.infer_ckpt(run))
cr.save(run)
cr.compare(run)
cr.board()

In [ ]:
# train R1 seed 2 (full fine-tuning)
run = cr.train("R1", 2)

In [ ]:
run = "R1_s2"
cr.score(run, cr.infer_ckpt(run))
cr.save(run)
cr.compare(run)
cr.board()